# 02 — Feature Engineering: Home Credit Default Risk

Tarea académica 2. Construimos la tabla de features a nivel `SK_ID_CURR` que va a alimentar el modelado supervisado (Tarea 4).

**Plan, basado directamente en los hallazgos de la Tarea 1 (`docs/informe-final.md`, sección 1):**

1. Limpieza de `application_train`: resolver la anomalía de `DAYS_EMPLOYED` (365243 → NaN + flag), y decidir tratamiento de nulos estructurales (`OWN_CAR_AGE`, variables de inmueble) vs. nulos a imputar.
2. Agregación de las 6 tablas relacionales a nivel `SK_ID_CURR`, generalizando los agregados que ya probamos en la EDA (conteos, sumas, ratios) y sumando estadísticas adicionales (min/max/mean/std) que en la EDA solo veíamos de forma puntual.
3. Ratios de negocio sobre `application_train` que el PRD (§7.2) señala como relevantes: credit-to-income, annuity-to-income, credit-to-goods.
4. Merge de todo en una única tabla de features por `SK_ID_CURR`, con un chequeo explícito de fuga de datos (ninguna feature debe usar información posterior a la fecha de la solicitud) y de cobertura (cuántos solicitantes quedan con NaN en cada bloque, ya relevado en la Tarea 1).
5. Persistir el resultado en `data/processed/` (no versionado en git, ver `.gitignore`).

In [1]:
import warnings
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw"
OUT_DIR = "../data/processed"

app_train = pd.read_csv(f"{DATA_DIR}/application_train.csv")
app_test = pd.read_csv(f"{DATA_DIR}/application_test.csv").assign(TARGET=np.nan)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", pd.errors.PerformanceWarning)
    app = pd.concat([app_train, app_test], ignore_index=True).assign(
        IS_TRAIN=lambda d: d["TARGET"].notna().astype(int)
    )

print(app.shape)
app[["IS_TRAIN", "TARGET"]].groupby("IS_TRAIN").agg(n=("TARGET", "size"), n_target_notna=("TARGET", "count"))

(356255, 123)


/var/folders/z1/f72k7b1x6qd890_dyn_vg6sr0000gn/T/ipykernel_46589/1916612721.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_test = pd.read_csv(f"{DATA_DIR}/application_test.csv").assign(TARGET=np.nan)


,n,n_target_notna
IS_TRAIN,,
0,48744,0
1,307511,307511


**Por qué concatenar `application_train` y `application_test` antes de generar features:** todas las agregaciones y ratios que vamos a construir dependen únicamente de `SK_ID_CURR` y de las tablas relacionales, nunca de `TARGET`. Si las calculáramos por separado en train y test, correríamos el riesgo de introducir inconsistencias (por ejemplo, una categoría rara que solo aparece en test y no se contempla al definir un encoding sobre train). Concatenar y separar por la columna `IS_TRAIN` al final es la forma estándar de evitar ese *train/test skew* sin tocar `TARGET`, que se mantiene `NaN` en test.

## Limpieza de `application_train`/`application_test`

Arrancamos por la anomalía de `DAYS_EMPLOYED` que ya identificamos en la Tarea 1: el valor centinela `365243` aparece cuando el solicitante no tiene relación de empleo activa (Pensioner/Unemployed). La reemplazamos por `NaN` y preservamos la señal con un flag booleano.

In [2]:
app["DAYS_EMPLOYED_ANOM"] = app["DAYS_EMPLOYED"] == 365243
app["DAYS_EMPLOYED"] = app["DAYS_EMPLOYED"].replace(365243, np.nan)
app = app.copy()

print(f"Filas marcadas como anómalas: {app['DAYS_EMPLOYED_ANOM'].sum()} de {len(app)}")
app["DAYS_EMPLOYED"].describe()

/var/folders/z1/f72k7b1x6qd890_dyn_vg6sr0000gn/T/ipykernel_46589/2506027459.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app["DAYS_EMPLOYED_ANOM"] = app["DAYS_EMPLOYED"] == 365243


Filas marcadas como anómalas: 64648 de 356255


count    291607.000000
mean      -2396.698937
std        2334.479967
min      -17912.000000
25%       -3200.000000
50%       -1663.000000
75%        -780.000000
max           0.000000
Name: DAYS_EMPLOYED, dtype: float64

## Ratios de negocio sobre `application_train`/`application_test`

El PRD (§7.2) señala credit-to-income, annuity-to-income y credit-to-goods como ratios de negocio relevantes en scoring crediticio: no es lo mismo deber un monto fijo con ingresos altos que con ingresos bajos, y la relación entre lo que se pide y el valor del bien financiado (`AMT_GOODS_PRICE`) indica cuánto "de más" se está pidiendo sobre el bien. Calculamos los tres y los cruzamos con `TARGET` (solo en la porción train) como chequeo rápido de señal, igual que hicimos con las tablas relacionales en la Tarea 1.

In [3]:
app["credit_to_income"] = app["AMT_CREDIT"] / app["AMT_INCOME_TOTAL"].replace(0, np.nan)
app["annuity_to_income"] = app["AMT_ANNUITY"] / app["AMT_INCOME_TOTAL"].replace(0, np.nan)
app["credit_to_goods"] = app["AMT_CREDIT"] / app["AMT_GOODS_PRICE"].replace(0, np.nan)

ratio_cols = ["credit_to_income", "annuity_to_income", "credit_to_goods"]
print(app[ratio_cols].describe())

train_mask = app["IS_TRAIN"] == 1
app.loc[train_mask, ratio_cols + ["TARGET"]].corr()["TARGET"]

       credit_to_income  annuity_to_income  credit_to_goods
count     356255.000000      356219.000000    355977.000000
mean           3.849476           0.181211         1.124213
std            2.635035           0.094701         0.124172
min            0.004808           0.000224         0.150000
25%            2.000000           0.114950         1.000000
50%            3.158857           0.163182         1.118800
75%            5.000000           0.229156         1.198000
max           84.736842           2.024714         6.000000


credit_to_income    -0.007727
annuity_to_income    0.014265
credit_to_goods      0.069427
TARGET               1.000000
Name: TARGET, dtype: float64

## Agregación de `bureau.csv`

En la Tarea 1 agregamos solo 3 variables puntuales (`bureau_count`, `bureau_active_count`, `bureau_debt_sum`) para verificar señal. Ahora generalizamos: agregamos min/max/mean/sum sobre las columnas de monto más relevantes (`AMT_CREDIT_SUM`, `AMT_CREDIT_SUM_DEBT`, `AMT_CREDIT_SUM_OVERDUE`, `AMT_ANNUITY`), además de conteos por estado (`CREDIT_ACTIVE`) que ya vimos concentrados en Closed/Active. Los solicitantes sin registro en `bureau.csv` (14.3%, según la Tarea 1) van a quedar en NaN tras el merge — eso ya está documentado como cobertura, no lo repetimos acá.

In [4]:
bureau = pd.read_csv(f"{DATA_DIR}/bureau.csv")

bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    bureau_count=("SK_ID_BUREAU", "count"),
    bureau_active_count=("CREDIT_ACTIVE", lambda s: (s == "Active").sum()),
    bureau_credit_sum_mean=("AMT_CREDIT_SUM", "mean"),
    bureau_credit_sum_max=("AMT_CREDIT_SUM", "max"),
    bureau_debt_sum=("AMT_CREDIT_SUM_DEBT", "sum"),
    bureau_debt_mean=("AMT_CREDIT_SUM_DEBT", "mean"),
    bureau_overdue_sum=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    bureau_overdue_max=("AMT_CREDIT_SUM_OVERDUE", "max"),
    bureau_annuity_mean=("AMT_ANNUITY", "mean"),
)

print(bureau_agg.shape)
bureau_agg.describe()

(305811, 9)


,bureau_count,bureau_active_count,bureau_credit_sum_mean,bureau_credit_sum_max,bureau_debt_sum,bureau_debt_mean,bureau_overdue_sum,bureau_overdue_max,bureau_annuity_mean
count,305811.000000,305811.000000,3.058090e+05,3.058090e+05,3.058110e+05,2.974390e+05,3.058110e+05,3.058110e+05,1.182240e+05
mean,5.612709,2.062081,3.807398e+05,9.917572e+05,6.539142e+05,1.616341e+05,2.127933e+02,1.764095e+02,1.560825e+04
std,4.430354,1.791724,8.792865e+05,2.317442e+06,1.640691e+06,5.367677e+05,1.569161e+04,1.368650e+04,1.524020e+05
min,1.000000,0.000000,0.000000e+00,0.000000e+00,-6.981558e+06,-1.083615e+06,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.000000,1.000000,1.039616e+05,1.800000e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,7.020000e+02
50%,4.000000,2.000000,1.972972e+05,4.500000e+05,1.733670e+05,4.476038e+04,0.000000e+00,0.000000e+00,6.516000e+03
75%,8.000000,3.000000,3.978647e+05,1.079100e+06,6.765412e+05,1.436004e+05,0.000000e+00,0.000000e+00,1.484335e+04
max,116.000000,32.000000,1.980723e+08,5.850000e+08,3.344983e+08,5.175000e+07,3.756681e+06,3.756681e+06,2.728243e+07


In [5]:
merged_bureau = app.loc[train_mask, ["SK_ID_CURR", "TARGET"]].merge(bureau_agg, on="SK_ID_CURR", how="left")
merged_bureau[list(bureau_agg.columns) + ["TARGET"]].corr()["TARGET"].sort_values()

bureau_credit_sum_mean   -0.019957
bureau_credit_sum_max    -0.019737
bureau_annuity_mean      -0.001391
bureau_debt_mean         -0.000637
bureau_count              0.004056
bureau_debt_sum           0.007144
bureau_overdue_max        0.010614
bureau_overdue_sum        0.013335
bureau_active_count       0.067128
TARGET                    1.000000
Name: TARGET, dtype: float64

## Agregación de `bureau_balance.csv`

Igual que en la Tarea 1, esta tabla requiere join de dos saltos (`bureau_balance` → `bureau` por `SK_ID_BUREAU` → `SK_ID_CURR`). Generalizamos: en vez de un único flag "algún mes en mora", agregamos por crédito la cantidad de meses en mora, el peor `STATUS` numérico alcanzado, y la cantidad total de meses de historial (proxy de antigüedad del crédito); después sumamos/promediamos por `SK_ID_CURR`.

In [6]:
bureau_balance = pd.read_csv(f"{DATA_DIR}/bureau_balance.csv")

status_to_dpd = {"C": 0, "X": 0, "0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5}
bureau_balance["dpd_level"] = bureau_balance["STATUS"].map(status_to_dpd)

bb_per_credit = bureau_balance.groupby("SK_ID_BUREAU").agg(
    months_count=("MONTHS_BALANCE", "count"),
    months_dpd=("dpd_level", lambda s: (s > 0).sum()),
    worst_dpd_level=("dpd_level", "max"),
)

bb_with_curr = bureau[["SK_ID_CURR", "SK_ID_BUREAU"]].merge(bb_per_credit, on="SK_ID_BUREAU", how="inner")

bb_agg = bb_with_curr.groupby("SK_ID_CURR").agg(
    bb_credits_with_history=("SK_ID_BUREAU", "count"),
    bb_months_dpd_sum=("months_dpd", "sum"),
    bb_worst_dpd_level_max=("worst_dpd_level", "max"),
    bb_months_count_mean=("months_count", "mean"),
)

print(bb_agg.shape)
bb_agg.describe()

(134542, 4)


,bb_credits_with_history,bb_months_dpd_sum,bb_worst_dpd_level_max,bb_months_count_mean
count,134542.000000,134542.000000,134542.000000,134542.000000
mean,5.755482,2.333398,0.514821,30.253224
std,4.556574,7.724338,0.932831,17.466661
min,1.000000,0.000000,0.000000,1.000000
25%,2.000000,0.000000,0.000000,17.000000
50%,5.000000,0.000000,0.000000,28.000000
75%,8.000000,2.000000,1.000000,40.888889
max,116.000000,396.000000,5.000000,97.000000


In [7]:
merged_bb = app.loc[train_mask, ["SK_ID_CURR", "TARGET"]].merge(bb_agg, on="SK_ID_CURR", how="left")
merged_bb[list(bb_agg.columns) + ["TARGET"]].corr()["TARGET"].sort_values()

bb_months_count_mean      -0.080193
bb_credits_with_history    0.006053
bb_months_dpd_sum          0.024829
bb_worst_dpd_level_max     0.035989
TARGET                     1.000000
Name: TARGET, dtype: float64

## Agregación de `previous_application.csv`

En la Tarea 1 vimos que `prev_approval_rate` era, hasta ese punto, la señal individual más fuerte entre las tablas relacionales. Generalizamos: sumamos estadísticas de monto (`AMT_CREDIT`, `AMT_ANNUITY`) y `days_decision_mean` (qué tan reciente/antigua es la actividad previa en promedio), además de las 3 variables ya validadas.

In [8]:
previous_application = pd.read_csv(f"{DATA_DIR}/previous_application.csv")

prev_agg = previous_application.groupby("SK_ID_CURR").agg(
    prev_count=("SK_ID_PREV", "count"),
    prev_ever_refused=("NAME_CONTRACT_STATUS", lambda s: (s == "Refused").any()),
    prev_approval_rate=("NAME_CONTRACT_STATUS", lambda s: (s == "Approved").mean()),
    prev_credit_mean=("AMT_CREDIT", "mean"),
    prev_annuity_mean=("AMT_ANNUITY", "mean"),
    prev_days_decision_mean=("DAYS_DECISION", "mean"),
)

merged_prev = app.loc[train_mask, ["SK_ID_CURR", "TARGET"]].merge(prev_agg, on="SK_ID_CURR", how="left")
print(prev_agg.shape)
merged_prev[list(prev_agg.columns) + ["TARGET"]].corr()["TARGET"].sort_values()

(338857, 6)


prev_approval_rate        -0.063521
prev_annuity_mean         -0.034871
prev_credit_mean          -0.016114
prev_count                 0.019762
prev_days_decision_mean    0.046864
prev_ever_refused          0.056291
TARGET                     1.000000
Name: TARGET, dtype: float64

## Agregación de `POS_CASH_balance.csv`

En la Tarea 1, `pos_ever_dpd`/`pos_dpd_max` (máximo/alguna vez en mora) dieron señal débil, contra la hipótesis planteada. Probamos la alternativa que quedó propuesta ahí: proporción de meses en mora (en vez de máximo/alguna vez), más cantidad de meses de historial y conteo de créditos POS/efectivo distintos.

In [9]:
pos_cash = pd.read_csv(f"{DATA_DIR}/POS_CASH_balance.csv")
pos_cash["dpd_flag"] = pos_cash["SK_DPD"] > 0

pos_agg = pos_cash.groupby("SK_ID_CURR").agg(
    pos_credits_count=("SK_ID_PREV", "nunique"),
    pos_months_count=("SK_ID_PREV", "count"),
    pos_dpd_rate=("dpd_flag", "mean"),
    pos_dpd_max=("SK_DPD", "max"),
)

merged_pos = app.loc[train_mask, ["SK_ID_CURR", "TARGET"]].merge(pos_agg, on="SK_ID_CURR", how="left")
print(pos_agg.shape)
merged_pos[list(pos_agg.columns) + ["TARGET"]].corr()["TARGET"].sort_values()

(337252, 4)


pos_credits_count   -0.040507
pos_months_count    -0.035632
pos_dpd_max          0.004763
pos_dpd_rate         0.030616
TARGET               1.000000
Name: TARGET, dtype: float64

## Agregación de `credit_card_balance.csv`

En la Tarea 1, `cc_utilization_mean` fue la señal relacional más fuerte del dataset (0.1356). Generalizamos sumando `cc_utilization_max`, `cc_dpd_rate` (mora) y `cc_cards_count` (tarjetas distintas).

**Por qué agregar el máximo de utilización, no solo el promedio:** el promedio mide uso *sostenido* del límite a lo largo de toda la relación con la tarjeta, pero puede diluir un pico puntual de estrés severo (alguien que usó el 95% del límite un par de meses por una emergencia y después volvió a un uso bajo tendría un promedio moderado). El máximo captura ese pico específico, independientemente de cuánto duró. Son señales de naturaleza distinta -uso sostenido alto (mean) sugiere dependencia estructural del crédito disponible, un pico aislado (max) sugiere un evento puntual de estrés- y no había forma de saber a priori cuál de las dos correlaciona más fuerte con `TARGET` sin probar ambas.

In [10]:
credit_card = pd.read_csv(f"{DATA_DIR}/credit_card_balance.csv")
credit_card["utilization_ratio"] = credit_card["AMT_BALANCE"] / credit_card["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
credit_card["dpd_flag"] = credit_card["SK_DPD"] > 0

cc_agg = credit_card.groupby("SK_ID_CURR").agg(
    cc_cards_count=("SK_ID_PREV", "nunique"),
    cc_utilization_mean=("utilization_ratio", "mean"),
    cc_utilization_max=("utilization_ratio", "max"),
    cc_dpd_rate=("dpd_flag", "mean"),
)

merged_cc = app.loc[train_mask, ["SK_ID_CURR", "TARGET"]].merge(cc_agg, on="SK_ID_CURR", how="left")
print(cc_agg.shape)
merged_cc[list(cc_agg.columns) + ["TARGET"]].corr()["TARGET"].sort_values()

(103558, 4)


cc_dpd_rate            0.001753
cc_cards_count         0.004388
cc_utilization_max     0.097011
cc_utilization_mean    0.135560
TARGET                 1.000000
Name: TARGET, dtype: float64

## Agregación de `installments_payments.csv`

En la Tarea 1, promediar atraso/déficit sobre todas las cuotas históricas dio la señal más débil del dataset relacional (0.0293/0.0209), contra la hipótesis de que sería la más fuerte por ser la más granular. Probamos las alternativas propuestas ahí: atraso máximo (en vez de promedio) y proporción de cuotas con atraso mayor a 5 días.

In [11]:
installments = pd.read_csv(f"{DATA_DIR}/installments_payments.csv")
installments["payment_delay"] = installments["DAYS_ENTRY_PAYMENT"] - installments["DAYS_INSTALMENT"]
installments["payment_shortfall"] = installments["AMT_INSTALMENT"] - installments["AMT_PAYMENT"]
installments["late_flag"] = installments["payment_delay"] > 5

inst_agg = installments.groupby("SK_ID_CURR").agg(
    inst_delay_mean=("payment_delay", "mean"),
    inst_delay_max=("payment_delay", "max"),
    inst_late_rate=("late_flag", "mean"),
    inst_shortfall_mean=("payment_shortfall", "mean"),
)

merged_inst = app.loc[train_mask, ["SK_ID_CURR", "TARGET"]].merge(inst_agg, on="SK_ID_CURR", how="left")
print(inst_agg.shape)
merged_inst[list(inst_agg.columns) + ["TARGET"]].corr()["TARGET"].sort_values()

(339587, 4)


inst_delay_max         0.004657
inst_delay_mean        0.020870
inst_shortfall_mean    0.029339
inst_late_rate         0.062513
TARGET                 1.000000
Name: TARGET, dtype: float64

## Merge final: tabla de features a nivel `SK_ID_CURR`

Unimos `app` (con los ratios de negocio ya calculados) con los 6 agregados relacionales (`bureau_agg`, `bb_agg`, `prev_agg`, `pos_agg`, `cc_agg`, `inst_agg`), todos con `how="left"` para no perder solicitantes sin registro en alguna tabla (esos casos quedan en NaN, tal como se documentó la cobertura de cada tabla en la Tarea 1). Chequeamos el shape final y que no se hayan duplicado filas por `SK_ID_CURR` en el proceso.

In [12]:
features = (
    app
    .merge(bureau_agg, on="SK_ID_CURR", how="left")
    .merge(bb_agg, on="SK_ID_CURR", how="left")
    .merge(prev_agg, on="SK_ID_CURR", how="left")
    .merge(pos_agg, on="SK_ID_CURR", how="left")
    .merge(cc_agg, on="SK_ID_CURR", how="left")
    .merge(inst_agg, on="SK_ID_CURR", how="left")
)

assert features["SK_ID_CURR"].is_unique, "SK_ID_CURR duplicado tras el merge"
assert len(features) == len(app), "el merge cambió la cantidad de filas"

print(features.shape)
new_feature_cols = (
    list(bureau_agg.columns) + list(bb_agg.columns) + list(prev_agg.columns)
    + list(pos_agg.columns) + list(cc_agg.columns) + list(inst_agg.columns)
)
print(f"Features relacionales agregadas: {len(new_feature_cols)}")
features[new_feature_cols].isnull().mean().sort_values(ascending=False)

(356255, 158)
Features relacionales agregadas: 31


cc_utilization_mean        0.712439
cc_utilization_max         0.712439
cc_dpd_rate                0.709315
cc_cards_count             0.709315
bureau_annuity_mean        0.668148
bb_worst_dpd_level_max     0.622344
bb_credits_with_history    0.622344
bb_months_dpd_sum          0.622344
bb_months_count_mean       0.622344
bureau_debt_mean           0.165095
bureau_credit_sum_mean     0.141601
bureau_credit_sum_max      0.141601
bureau_active_count        0.141595
bureau_count               0.141595
bureau_overdue_max         0.141595
bureau_overdue_sum         0.141595
bureau_debt_sum            0.141595
pos_dpd_max                0.053341
pos_credits_count          0.053341
pos_months_count           0.053341
pos_dpd_rate               0.053341
prev_annuity_mean          0.050183
prev_approval_rate         0.048836
prev_days_decision_mean    0.048836
prev_credit_mean           0.048836
prev_ever_refused          0.048836
prev_count                 0.048836
inst_delay_mean            0

## Chequeo intermedio (sin persistir todavía)

Antes de guardar, todavía faltan los flags de "sin registro" y el encoding de categóricas (siguientes celdas). El chequeo de fuga de datos del plan original queda cubierto por construcción: todas las variables agregadas provienen de columnas que ya existían en las tablas fuente antes de la fecha de la solicitud actual (`DAYS_*` son siempre negativos o relativos a la solicitud), y ninguna agregación usó `TARGET` como insumo.

In [13]:
assert features["SK_ID_CURR"].is_unique
print(features.shape)

(356255, 158)


## Estrategia de imputación

**Decisión de diseño:** el modelo primario del proyecto es XGBoost, que maneja `NaN` nativamente: en cada split, aprende hacia qué rama mandar los valores faltantes como parte del entrenamiento, en vez de requerir un valor sustituto. Imputar a ciegas antes de XGBoost destruiría la señal de "sin registro" que ya identificamos como informativa en la Tarea 1 (por ejemplo, el 72% de solicitantes sin `credit_card_balance` no es un dato faltante al azar, es "esta persona no tuvo tarjeta de crédito con Home Credit").

Por eso: (1) para XGBoost, dejamos los `NaN` como están y solo agregamos flags booleanos explícitos de "sin registro" por tabla relacional, que ya son una feature en sí misma más allá de lo que el árbol infiera del NaN; (2) para el baseline de Regresión Logística (que sí requiere imputación numérica), la resolvemos más adelante con un `Pipeline`/`ColumnTransformer` de Scikit-learn específico para ese modelo, sin tocar la tabla de features cruda que usa XGBoost.

In [14]:
no_record_specs = {
    "bureau_no_record": "bureau_count",
    "bb_no_record": "bb_credits_with_history",
    "prev_no_record": "prev_count",
    "pos_no_record": "pos_credits_count",
    "cc_no_record": "cc_cards_count",
    "inst_no_record": "inst_delay_mean",
}

for flag_col, ref_col in no_record_specs.items():
    features[flag_col] = features[ref_col].isna()

flag_cols = list(no_record_specs.keys())
print(features[flag_cols].mean())

bureau_no_record    0.141595
bb_no_record        0.622344
prev_no_record      0.048836
pos_no_record       0.053341
cc_no_record        0.709315
inst_no_record      0.046812
dtype: float64


## Encoding de variables categóricas

XGBoost (a diferencia de scikit-learn's `DecisionTree`) tampoco acepta strings directamente, así que las categóricas de `application_train`/`application_test` necesitan codificarse igual. Miramos primero la cardinalidad de cada columna categórica para decidir la estrategia: one-hot para las de baja cardinalidad, revisar caso por caso las de alta cardinalidad.

In [15]:
cat_cols = features.select_dtypes(include=["object", "str"]).columns.tolist()
cardinality = features[cat_cols].nunique().sort_values(ascending=False)
print(f"Columnas categóricas: {len(cat_cols)}")
cardinality

Columnas categóricas: 17


ORGANIZATION_TYPE             58
OCCUPATION_TYPE               18
NAME_INCOME_TYPE               8
WALLSMATERIAL_MODE             7
NAME_TYPE_SUITE                7
WEEKDAY_APPR_PROCESS_START     7
NAME_HOUSING_TYPE              6
NAME_FAMILY_STATUS             6
NAME_EDUCATION_TYPE            5
FONDKAPREMONT_MODE             4
HOUSETYPE_MODE                 3
CODE_GENDER                    3
EMERGENCYSTATE_MODE            2
NAME_CONTRACT_TYPE             2
FLAG_OWN_REALTY                2
FLAG_OWN_CAR                   2
prev_ever_refused              2
dtype: int64

`prev_ever_refused` quedó tipada como `object` por el `NaN` que introduce el `left merge` sobre un booleano (los solicitantes sin `previous_application`): no es una categórica real, es un flag de 3 estados (True/False/sin dato). La tratamos aparte, no con one-hot.

Las 16 categóricas restantes tienen cardinalidad manejable (máximo 58, `ORGANIZATION_TYPE`), así que aplicamos one-hot a todas: para árboles (XGBoost) el volumen de columnas no es un problema de la misma magnitud que para modelos lineales, y mantenemos la información completa en vez de agrupar categorías raras a priori.

In [16]:
onehot_cols = [c for c in cat_cols if c != "prev_ever_refused"]

features["prev_ever_refused"] = features["prev_ever_refused"].map({True: 1, False: 0})

n_cols_before = features.shape[1]
features = pd.get_dummies(features, columns=onehot_cols, dummy_na=True)
n_cols_after = features.shape[1]

print(f"Columnas antes del one-hot: {n_cols_before}")
print(f"Columnas después del one-hot: {n_cols_after}")
print(f"Columnas nuevas por one-hot: {n_cols_after - n_cols_before}")

Columnas antes del one-hot: 164
Columnas después del one-hot: 304
Columnas nuevas por one-hot: 140


## Persistencia final

Guardamos la tabla de features completa (limpieza + ratios + agregaciones relacionales + flags de "sin registro" + one-hot) en `data/processed/features.parquet`.

In [17]:
import os

os.makedirs(OUT_DIR, exist_ok=True)
features.to_parquet(f"{OUT_DIR}/features.parquet", index=False)
print(f"Guardado en {OUT_DIR}/features.parquet: {features.shape}")

Guardado en ../data/processed/features.parquet: (356255, 304)
